# Precision Transformer — GPU Training Notebook

**Goal:** train a regime-aware sequence model on long-horizon 1m data using Colab GPU. Outputs a model that explicitly learns regime transitions and adjusts its predictions accordingly.

## Architecture

**`PrecisionTransformer`** — multi-task encoder with regime-aware design:

```
Tabular features (47-dim, 90-bar sequence)
     │
     ▼
Linear projection → 128-dim
     │
     ▼
+ Sinusoidal positional encoding
     │
     ▼
Transformer encoder (6 layers, 8 heads, FFN=512, dropout=0.2)
     │
     ▼
Last-token + mean-pool concatenation (256-dim)
     │
     ├─→ Signal head      (3-class: short / hold / long)
     ├─→ Regime head     (n_regime-class — auxiliary, helps backbone learn regime structure)
     └─→ Vol regression   (next-bar realised volatility — auxiliary)

Loss = signal_ce + 0.3 * regime_ce + 0.2 * vol_mse
```

Multi-task heads share the backbone. The regime auxiliary task forces the backbone to encode regime transitions explicitly, which the signal head then exploits.

## Training

- Mixed precision (FP16) — ~2x speedup on T4/L4/A100
- Cosine LR schedule with linear warmup
- AdamW + weight decay 0.01
- Gradient clipping at 1.0
- Class weights via inverse frequency
- Temporal decay sample weights (recency-aware)
- Early stopping on validation F1
- Max 200 epochs (typical convergence ~40-80)

## Honest expectations

- Tree ensembles (XGBoost/CatBoost stacking) typically match or beat single-Transformer on 1m FX/gold/crypto directional accuracy (Borrageiro 2022, Wang 2024, Tashiro 2023).
- Where Transformers DO win: regime-transition prediction, volatility forecasting, longer-horizon (5m+) signals.
- Run the smoke test (Section 4) — gates further training on real OOS performance, not training-loss.
- If WR ≤ 50% on smoke, abort. Adding more training time won't help — it's a data/architecture-fit problem, not an optimisation problem.

## 1. Environment + GPU check

In [ ]:
import os, sys, subprocess

# ── Edit this to your repo URL or Drive path ──
REPO_URL = "https://github.com/YOUR_USERNAME/vibetrading.git"
REPO_DIR = "/content/vibetrading"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
FRONTEND = os.path.join(REPO_DIR, "trading_terminal", "frontend")
sys.path.insert(0, FRONTEND)
os.chdir(FRONTEND)
print("Frontend on path:", FRONTEND)

In [ ]:
%pip install -q torch torchmetrics scikit-learn pandas numpy yfinance \
    pykalman hmmlearn xgboost lightgbm joblib

In [ ]:
import warnings; warnings.filterwarnings("ignore")
os.environ["ENABLE_TRADING_MEMORY"] = "false"
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"FP16 supported: {torch.cuda.is_bf16_supported() or True}")
else:
    print("⚠️  No GPU detected — switch Runtime → Change runtime type → GPU")

In [ ]:
from services.precision_trading_system import (
    PrecisionTradingSystem, Asset,
    triple_barrier_labels_vectorized,
    MarkovRegimeForecaster,
)
from services.training_pipeline import DataCleaner
print("Repo modules imported")

## 2. Data loading

For a long-horizon GPU run, more data = more value. **Strongly recommend Dukascopy** (free, 1m FX/gold/CFDs going back 10+ years). yfinance 1m is capped at 30 days. Pick ONE cell.

In [ ]:
# ── 2A: Upload your own large CSV ──
from google.colab import files
uploaded = files.upload()
csv_name = next(iter(uploaded.keys()))
df = pd.read_csv(csv_name)
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.set_index("timestamp")
df = df[["open", "high", "low", "close", "volume"]].sort_index().dropna()
print(f"Loaded {len(df):,} bars")

In [ ]:
# ── 2B: Dukascopy — years of free 1m FX / gold / index data ──
%pip install -q dukascopy_python
from datetime import datetime, timedelta
from dukascopy_python import (
    fetch, INTERVAL_MIN_1, OFFER_SIDE_BID,
)
from dukascopy_python.instruments import (
    INSTRUMENT_CFD_XAU_USD,                    # gold
    INSTRUMENT_FX_MAJORS_EUR_USD,
    INSTRUMENT_FX_MAJORS_GBP_USD,
)

INSTR = INSTRUMENT_CFD_XAU_USD
DAYS = 730                       # 2 years; bump higher if you want more
end = datetime.now()
start = end - timedelta(days=DAYS)

df = fetch(INSTR, INTERVAL_MIN_1, OFFER_SIDE_BID, start, end)
df.columns = [c.lower() for c in df.columns]
df = df[["open", "high", "low", "close", "volume"]].dropna()
if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")
print(f"Dukascopy loaded {len(df):,} bars · {df.index[0]} → {df.index[-1]}")

In [ ]:
# ── 2C: Repo's prebuilt CSV (60-day 1m or longer 1h) ──
PAIR = "XAUUSD"; TIMEFRAME = "1m"
csv = f"data/historical/{PAIR}_{TIMEFRAME}.csv"
df = pd.read_csv(csv, index_col="timestamp", parse_dates=["timestamp"])
if "source" in df.columns:
    df = df[df["source"] == "yfinance"]
df = df[["open", "high", "low", "close", "volume"]].sort_index().dropna()
if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")
print(f"Repo CSV loaded {len(df):,} bars · {df.index[0]} → {df.index[-1]}")

## 3. Feature engineering + sequence dataset

Reuses the repo's `_build_features` (47-dim feature set: VPIN, CVD, Volume Profile, FVG, MTF, Markov, microstructure). Then converts to overlapping fixed-length sequences for the Transformer.

In [ ]:
# Clean data first
cleaner = DataCleaner()
df = cleaner.remove_duplicates(df)
df = cleaner.fill_gaps(df, max_gap_minutes=5, interval="1m")
df, n_outliers = cleaner.remove_outliers(df, zscore=4.0)
print(f"Post-clean: {len(df):,} bars · {n_outliers} outliers removed")

ASSET = "XAUUSD"
sys_obj = PrecisionTradingSystem(asset=Asset(ASSET), model_type="xgboost", use_hmm=True)
feats = sys_obj._build_features(df)
print(f"Engineered features: {len(feats):,} bars × {feats.shape[1]} cols")
print("Sample columns:", list(feats.columns)[:10])

In [ ]:
# Triple-barrier labels (FIXED, symmetric — same as repo)
PT_MULT = float(getattr(sys_obj.config, "atr_multiplier_tp1", 1.5))
SL_MULT = float(getattr(sys_obj.config, "atr_multiplier_stop", 1.0))
MAX_BARS = 10
labels = triple_barrier_labels_vectorized(
    feats, pt_atr_mult=PT_MULT, sl_atr_mult=SL_MULT, max_bars=MAX_BARS,
)
print(f"Label dist: {labels.value_counts().sort_index().to_dict()}")

# Auxiliary regime label (for multi-task learning)
regime = feats.get("regime", pd.Series(0, index=feats.index)).fillna(0).astype(int)
n_regimes = int(regime.max() + 1) if regime.max() >= 0 else 2
print(f"Regimes: {n_regimes} (dist: {regime.value_counts().sort_index().to_dict()})")

# Auxiliary volatility regression target (next-bar realised vol)
fwd_vol = feats["close"].pct_change(5).rolling(5).std().shift(-5).fillna(0)

In [ ]:
# Drop OHLCV from feature matrix (keep only engineered features)
drop = ["open", "high", "low", "close", "volume"]
feature_cols = [c for c in feats.columns if c not in drop]
X = feats[feature_cols].fillna(0).values.astype(np.float32)
y_signal = labels.values.astype(np.int64)
y_regime = regime.values.astype(np.int64)
y_vol = fwd_vol.values.astype(np.float32)

# Standardise features (fit on train only — done in dataset class below)
print(f"X: {X.shape}, y_signal: {y_signal.shape}, y_regime: {y_regime.shape}")
print(f"Signal label counts: {pd.Series(y_signal).value_counts().sort_index().to_dict()}")

In [ ]:
SEQ_LEN = 90        # 90-bar lookback (~1.5h on 1m)
BATCH_SIZE = 128

class SequenceDataset(Dataset):
    """Rolling-window sequences over feature matrix with multi-task targets."""
    def __init__(self, X, y_sig, y_reg, y_vol, seq_len, weights=None):
        self.X = X; self.y_sig = y_sig; self.y_reg = y_reg; self.y_vol = y_vol
        self.seq_len = seq_len
        self.weights = weights if weights is not None else np.ones(len(X), dtype=np.float32)
    def __len__(self):
        return max(0, len(self.X) - self.seq_len)
    def __getitem__(self, i):
        seq = self.X[i:i + self.seq_len]              # (T, F)
        sig = self.y_sig[i + self.seq_len - 1] + 1    # {-1,0,1} → {0,1,2}
        reg = self.y_reg[i + self.seq_len - 1]
        vol = self.y_vol[i + self.seq_len - 1]
        w = self.weights[i + self.seq_len - 1]
        return (
            torch.from_numpy(seq), torch.tensor(sig, dtype=torch.long),
            torch.tensor(reg, dtype=torch.long), torch.tensor(vol, dtype=torch.float32),
            torch.tensor(w, dtype=torch.float32),
        )

n = len(X)
test_n = int(n * 0.10)
val_n = int(n * 0.10)
train_end = n - test_n - val_n

scaler = StandardScaler()
X_train = scaler.fit_transform(X[:train_end])
X_val = scaler.transform(X[train_end:train_end + val_n])
X_test = scaler.transform(X[train_end + val_n:])

# Temporal decay weights — recent bars dominate
halflife = max(720, train_end // 200)  # asset-aware
decay_lambda = np.log(2) / halflife
t = np.arange(train_end)
w_train = np.exp(decay_lambda * (t - train_end)).astype(np.float32)
w_train /= w_train.mean()

ds_train = SequenceDataset(X_train, y_signal[:train_end], y_regime[:train_end],
                            y_vol[:train_end], SEQ_LEN, w_train)
ds_val = SequenceDataset(X_val, y_signal[train_end:train_end + val_n],
                          y_regime[train_end:train_end + val_n],
                          y_vol[train_end:train_end + val_n], SEQ_LEN)
ds_test = SequenceDataset(X_test, y_signal[train_end + val_n:],
                           y_regime[train_end + val_n:],
                           y_vol[train_end + val_n:], SEQ_LEN)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Sequences — train: {len(ds_train)}, val: {len(ds_val)}, test: {len(ds_test)}")

## 4. Smoke test on a small subset (gate)

Run a 10-epoch training on the first 10k sequences. If OOS WR ≤ 50%, the long run won't help — abort and tweak data / labels / horizon instead.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class PrecisionTransformer(nn.Module):
    """Multi-task regime-aware Transformer encoder."""
    def __init__(self, n_features, n_regimes, d_model=128, nhead=8,
                 num_layers=6, dim_ff=512, dropout=0.2, n_signal_classes=3):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pe = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        # Multi-task heads share backbone
        head_dim = d_model * 2  # last + mean pool
        self.signal_head = nn.Sequential(
            nn.Linear(head_dim, head_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(head_dim // 2, n_signal_classes),
        )
        self.regime_head = nn.Sequential(
            nn.Linear(head_dim, head_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(head_dim // 2, n_regimes),
        )
        self.vol_head = nn.Sequential(
            nn.Linear(head_dim, head_dim // 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(head_dim // 4, 1),
        )
    def forward(self, x):                                # x: (B, T, F)
        h = self.input_proj(x)                            # (B, T, D)
        h = self.pe(h)
        h = self.encoder(h)
        h = self.norm(h)
        last = h[:, -1, :]                                # (B, D)
        mean = h.mean(dim=1)                              # (B, D)
        pooled = torch.cat([last, mean], dim=-1)          # (B, 2D)
        return self.signal_head(pooled), self.regime_head(pooled), self.vol_head(pooled).squeeze(-1)


def make_model():
    return PrecisionTransformer(
        n_features=X.shape[1], n_regimes=max(2, n_regimes),
        d_model=128, nhead=8, num_layers=6, dim_ff=512, dropout=0.2,
    ).to(DEVICE)

model = make_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"PrecisionTransformer: {n_params:,} params ({n_params * 4 / 1024**2:.1f} MB FP32)")

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# Class weights via inverse frequency (signal head only — regimes are balanced)
cls_counts = np.bincount(y_signal[:train_end] + 1, minlength=3) + 1
cls_w = 1.0 / cls_counts
cls_w = cls_w / cls_w.sum() * 3
cls_w_t = torch.tensor(cls_w, dtype=torch.float32, device=DEVICE)
print(f"Class weights (short/hold/long): {cls_w}")


def train_epoch(model, loader, opt, scaler_amp, w_signal=1.0, w_regime=0.3, w_vol=0.2):
    model.train()
    losses = []
    for seq, sig, reg, vol, sw in loader:
        seq = seq.to(DEVICE, non_blocking=True)
        sig = sig.to(DEVICE, non_blocking=True)
        reg = reg.to(DEVICE, non_blocking=True)
        vol = vol.to(DEVICE, non_blocking=True)
        sw = sw.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(enabled=DEVICE.type == "cuda"):
            ls, lr_, lv = model(seq)
            ce_sig = F.cross_entropy(ls, sig, weight=cls_w_t, reduction="none")
            ce_reg = F.cross_entropy(lr_, reg, reduction="none")
            mse_vol = F.mse_loss(lv, vol, reduction="none")
            loss = (sw * (w_signal * ce_sig + w_regime * ce_reg + w_vol * mse_vol)).mean()
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(opt)
        scaler_amp.update()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, targs, regs_pred, regs_true = [], [], [], []
    for seq, sig, reg, _, _ in loader:
        seq = seq.to(DEVICE, non_blocking=True)
        with autocast(enabled=DEVICE.type == "cuda"):
            ls, lr_, _ = model(seq)
        p = ls.argmax(dim=-1).cpu().numpy()
        preds.append(p); targs.append(sig.numpy())
        regs_pred.append(lr_.argmax(dim=-1).cpu().numpy()); regs_true.append(reg.numpy())
    p = np.concatenate(preds); t = np.concatenate(targs)
    rp = np.concatenate(regs_pred); rt = np.concatenate(regs_true)
    f1 = f1_score(t, p, average="macro", zero_division=0)
    acc = (p == t).mean()
    nz = p != 1                            # 1 = HOLD
    wr = (np.sign(t[nz] - 1) == np.sign(p[nz] - 1)).mean() if nz.sum() else 0.0
    mcc = matthews_corrcoef(t, p) if len(np.unique(t)) > 1 else 0.0
    reg_acc = (rp == rt).mean()
    return {"f1": f1, "acc": acc, "wr": wr, "mcc": mcc, "regime_acc": reg_acc,
            "pred_dist": pd.Series(p).value_counts().sort_index().to_dict()}


def train_n_epochs(epochs, lr=2e-4, smoke=False):
    model = make_model()
    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    scaler_amp = GradScaler(enabled=DEVICE.type == "cuda")
    best_val_f1 = 0.0; best_state = None; patience = 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl = train_epoch(model, dl_train, opt, scaler_amp)
        sched.step()
        if ep % 2 == 0 or ep == epochs:
            vm = evaluate(model, dl_val)
            print(f"  ep {ep:3d}  loss={tl:.4f}  val F1={vm['f1']:.3f} acc={vm['acc']:.1%} "
                  f"wr={vm['wr']:.1%} reg_acc={vm['regime_acc']:.1%}  ({time.time()-t0:.1f}s)")
            if vm["f1"] > best_val_f1:
                best_val_f1 = vm["f1"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 6 and not smoke:
                    print(f"  early stopping at epoch {ep}")
                    break
    if best_state:
        model.load_state_dict(best_state)
    return model, best_val_f1


# ── SMOKE TEST: 10 epochs ──
print("\n── SMOKE TEST (10 epochs) ──")
smoke_model, smoke_val_f1 = train_n_epochs(epochs=10, lr=2e-4, smoke=True)
smoke_test = evaluate(smoke_model, dl_test)
print(f"\nSmoke test OOS:")
for k, v in smoke_test.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Smoke gate ──
SMOKE_WR = 0.50
SMOKE_F1 = 0.20
PROCEED = smoke_test["wr"] >= SMOKE_WR and smoke_test["f1"] >= SMOKE_F1
if not PROCEED:
    print("❌ SMOKE FAILED — abort. Adjust data / labels / horizon, not training time.")
    raise SystemExit("smoke failed")
print(f"✅ SMOKE PASSED — proceeding to full training")

## 5. Full GPU training

200 epoch budget (with early stopping, typically converges 40-80). Cosine LR schedule, AdamW, mixed precision, gradient clipping. Multi-task loss with regime + vol auxiliary signals.

In [ ]:
FULL_EPOCHS = 200
LR = 3e-4

print(f"\n── FULL TRAINING — {FULL_EPOCHS} epochs, lr={LR}, FP16={DEVICE.type=='cuda'} ──")
t0 = time.time()
model_full, best_val_f1 = train_n_epochs(epochs=FULL_EPOCHS, lr=LR, smoke=False)
print(f"\nDone in {time.time()-t0:.1f}s · best val F1: {best_val_f1:.3f}")

test_metrics = evaluate(model_full, dl_test)
print("\n── FINAL OOS RESULTS ──")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

## 6. Compare against XGBoost baseline (sanity check)

Trains a single XGBoost on flat features (no sequence) — same data split, same labels. If the Transformer doesn't beat XGBoost by ≥1pp WR, ship the simpler model.

In [ ]:
import xgboost as xgb

X_tr_flat = X[:train_end]
X_va_flat = X[train_end:train_end + val_n]
X_te_flat = X[train_end + val_n:]
y_tr = y_signal[:train_end] + 1
y_va = y_signal[train_end:train_end + val_n] + 1
y_te = y_signal[train_end + val_n:] + 1

scaler_xgb = StandardScaler().fit(X_tr_flat)
xgb_model = xgb.XGBClassifier(
    n_estimators=2000, max_depth=6, learning_rate=0.01,
    reg_lambda=5.0, min_child_weight=20, subsample=0.9, colsample_bytree=0.9,
    tree_method="hist", device="cuda" if DEVICE.type == "cuda" else "cpu",
    early_stopping_rounds=100, eval_metric="mlogloss", random_state=42,
)
xgb_model.fit(scaler_xgb.transform(X_tr_flat), y_tr,
              eval_set=[(scaler_xgb.transform(X_va_flat), y_va)], verbose=False)
y_pred_xgb = xgb_model.predict(scaler_xgb.transform(X_te_flat))
xgb_f1 = f1_score(y_te, y_pred_xgb, average="macro", zero_division=0)
xgb_acc = (y_pred_xgb == y_te).mean()
nz = y_pred_xgb != 1
xgb_wr = (np.sign(y_te[nz] - 1) == np.sign(y_pred_xgb[nz] - 1)).mean() if nz.sum() else 0.0

print("            F1     Acc     WR")
print(f"XGBoost   {xgb_f1:.3f}  {xgb_acc:.1%}  {xgb_wr:.1%}")
print(f"Trans.    {test_metrics['f1']:.3f}  {test_metrics['acc']:.1%}  {test_metrics['wr']:.1%}")
lift_wr = test_metrics['wr'] - xgb_wr
lift_f1 = test_metrics['f1'] - xgb_f1
print(f"\nLift: WR {lift_wr:+.1%}  F1 {lift_f1:+.3f}")
USE_TRANSFORMER = lift_wr >= 0.01
FINAL = "transformer" if USE_TRANSFORMER else "xgboost"
print(f"→ Recommend: {FINAL}")

## 7. Save bundle + download

Bundle = model state_dict + scaler + label map + config + metrics. Uses `torch.save` for the full model. Bundle is self-contained — load anywhere with PyTorch.

In [ ]:
import joblib
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"/content/precision_models_{ASSET}"
os.makedirs(out_dir, exist_ok=True)

if USE_TRANSFORMER:
    bundle_path = f"{out_dir}/precision_{ASSET}_transformer_{ts}.pt"
    torch.save({
        "model_state": model_full.state_dict(),
        "model_config": dict(
            n_features=X.shape[1], n_regimes=max(2, n_regimes),
            d_model=128, nhead=8, num_layers=6, dim_ff=512, dropout=0.2,
            n_signal_classes=3, seq_len=SEQ_LEN,
        ),
        "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_,
        "feature_cols": feature_cols,
        "label_map": {-1: 0, 0: 1, 1: 2}, "inv_map": {0: -1, 1: 0, 2: 1},
        "asset": ASSET, "trained_at": datetime.now().isoformat(),
        "metrics_test": test_metrics, "metrics_xgb": {"f1": float(xgb_f1), "acc": float(xgb_acc), "wr": float(xgb_wr)},
        "data_range": [str(df.index[0]), str(df.index[-1])],
        "n_train": int(train_end), "n_val": int(val_n), "n_test": int(test_n),
    }, bundle_path)
else:
    bundle_path = f"{out_dir}/precision_{ASSET}_xgboost_{ts}.pkl"
    joblib.dump({
        "xgb_model": xgb_model, "scaler": scaler_xgb,
        "feature_cols": feature_cols, "label_map": {-1: 0, 0: 1, 1: 2}, "inv_map": {0: -1, 1: 0, 2: 1},
        "asset": ASSET, "trained_at": datetime.now().isoformat(),
        "metrics_test": {"f1": float(xgb_f1), "acc": float(xgb_acc), "wr": float(xgb_wr)},
    }, bundle_path)

size_mb = os.path.getsize(bundle_path) / 1024 / 1024
print(f"Saved: {bundle_path} ({size_mb:.1f} MB)")
from google.colab import files
files.download(bundle_path)

## 8. In-notebook test on the saved bundle

Reloads the bundle to confirm it deserialises and predicts as expected.

In [ ]:
if USE_TRANSFORMER:
    ck = torch.load(bundle_path, map_location=DEVICE)
    m = PrecisionTransformer(**{k: v for k, v in ck["model_config"].items() if k != "seq_len"}).to(DEVICE)
    m.load_state_dict(ck["model_state"])
    m.eval()
    sc_mean, sc_scale = ck["scaler_mean"], ck["scaler_scale"]
    seq_len = ck["model_config"]["seq_len"]
    # Predict on last 200 sequences
    test_preds = []
    with torch.no_grad():
        for i in range(min(200, len(ds_test))):
            seq, sig, reg, _, _ = ds_test[i]
            seq = seq.unsqueeze(0).to(DEVICE)
            ls, lr_, lv = m(seq)
            test_preds.append(int(ls.argmax(dim=-1).item()))
    print("Transformer last-200 prediction distribution:")
    print(pd.Series([ck["inv_map"][p] for p in test_preds]).value_counts().sort_index().to_dict())
else:
    bundle = joblib.load(bundle_path)
    p = bundle["xgb_model"].predict(bundle["scaler"].transform(X_te_flat[:200]))
    p_orig = [bundle["inv_map"][int(x)] for x in p]
    print("XGBoost last-200 prediction distribution:")
    print(pd.Series(p_orig).value_counts().sort_index().to_dict())

## What to expect in practice

On 1m XAU/EUR/BTC, with 1-2 years of data:

| Metric | Realistic range | What you have |
|---|---|---|
| OOS F1 | 0.20-0.45 | (above) |
| OOS Win-Rate | 50-60% | (above) |
| OOS Acc (3-class) | 40-55% | (above) |
| Regime Acc | 70-90% | (above) |
| MCC | 0.05-0.20 | (above) |

If WR < 50%: data isn't right. Try (a) longer history (Dukascopy 5+ yrs), (b) different label horizon (5 / 20 bars), (c) symmetric `pt=sl`, (d) different asset (BTC has more 1m signal than EUR).

If F1 < 0.2 but WR > 55%: model is over-confident on rare classes. Reduce `n_signal_classes` to 3 (already done) and increase regime aux loss weight.

## To use the trained model in the local repo

1. Download the `.pt` file from Colab.
2. Place it in `data/models/<ASSET>/`.
3. Add a `TransformerSignalModel` adapter in `services/precision_trading_system.py` that wraps the `PrecisionTransformer` definition + `state_dict` load + `scaler` + sequence buffering for live inference.
4. Wire it as a new `model_type="transformer"` option in `PrecisionTradingSystem.__init__`.